## Import libraries

In [450]:
import pandas as pd
import json
import random
from config import RANDOM_SEED
import ast

random.seed(RANDOM_SEED)

## Training data preparation 

### 1. Load the necessary files

In [451]:
job_skill_match = pd.read_csv('./data/raw/TaskB/training/job2skill.tsv', header=None, sep="\t", names=['job_id', 'skill_id', 'label'])

In [452]:
with open('./data/raw/TaskB/training/jobid2terms.json', 'r', encoding='utf-8') as f:
    jobs_dict = json.load(f)

jobs = pd.DataFrame(data={ 'job_id': list(jobs_dict.keys()), 'job_names': list(jobs_dict.values()) })

In [453]:
with open('./data/raw/TaskB/training/skillid2terms.json', 'r', encoding='utf-8') as f:
    skills_dict = json.load(f)

skills = pd.DataFrame(data={ 'skill_id': list(skills_dict.keys()), 'skill_names': list(skills_dict.values()) })

### 2. Check the data

#### 2.1 job_skill_match

In [454]:
job_skill_match.head()

,job_id,skill_id,label
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [455]:
job_skill_match.info()

<class 'pandas.DataFrame'>
RangeIndex: 114699 entries, 0 to 114698
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   job_id    114699 non-null  str  
 1   skill_id  114699 non-null  str  
 2   label     114699 non-null  str  
dtypes: str(3)
memory usage: 2.6 MB


In [456]:
job_skill_match.duplicated().sum()

np.int64(0)

In [457]:
job_skill_match['job_id'].unique().size

3011

In [458]:
job_skill_match['skill_id'].unique().size

13224

In [459]:
job_skill_match['label'].value_counts()

label
essential    62480
optional     52219
Name: count, dtype: int64

#### 2.2 jobs

In [460]:
jobs.head()

,job_id,job_names
0,http://data.europa.eu/esco/occupation/00030d09...,"[technical director, technical and operations ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,"[metal drawing machine operator, metal drawing..."
2,http://data.europa.eu/esco/occupation/0019b951...,"[precision device inspector, inspector of prec..."
3,http://data.europa.eu/esco/occupation/0022f466...,"[air traffic safety technician, air traffic sa..."
4,http://data.europa.eu/esco/occupation/002da35b...,"[hospitality revenue manager, hospitality reve..."


In [461]:
jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 3039 entries, 0 to 3038
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   job_id     3039 non-null   str   
 1   job_names  3039 non-null   object
dtypes: object(1), str(1)
memory usage: 47.6+ KB


In [462]:
jobs['job_id'].unique().size

3039

#### 2.3 skills

In [463]:
skills.head()

,skill_id,skill_names
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,"[manage musical staff, manage staff of music, ..."
1,http://data.europa.eu/esco/skill/00064735-8fad...,"[supervise correctional procedures, oversee pr..."
2,http://data.europa.eu/esco/skill/000709ed-2be5...,"[apply anti-oppressive practices, apply non-op..."
3,http://data.europa.eu/esco/skill/0007bdc2-dd15...,[control compliance of railway vehicles regula...
4,http://data.europa.eu/esco/skill/00090cc1-1f27...,"[identify available services, establish availa..."


In [464]:
skills.info()

<class 'pandas.DataFrame'>
RangeIndex: 13939 entries, 0 to 13938
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   skill_id     13939 non-null  str   
 1   skill_names  13939 non-null  object
dtypes: object(1), str(1)
memory usage: 217.9+ KB


In [465]:
skills['skill_id'].unique().size

13939

### 3. Creating the training dataset

In [466]:
jobs['job_name'] = jobs['job_names'].apply( lambda x: x[0] )
skills['skill_name'] = skills['skill_names'].apply( lambda x: x[0] )

In [467]:
train = pd.merge( left=job_skill_match, right=jobs[['job_id', 'job_name']], how='inner', on='job_id' )
train = pd.merge( left=train, right=skills[['skill_id', 'skill_name']], how='inner', on='skill_id' )

In [468]:
train.head()

,job_id,skill_id,label,job_name,skill_name
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential,technical director,promote health and safety
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential,technical director,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential,technical director,negotiate health and safety issues with third ...
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential,technical director,theatre techniques
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential,technical director,coordinate technical teams in artistic product...


In [469]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 114699 entries, 0 to 114698
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   job_id      114699 non-null  str  
 1   skill_id    114699 non-null  str  
 2   label       114699 non-null  str  
 3   job_name    114699 non-null  str  
 4   skill_name  114699 non-null  str  
dtypes: str(5)
memory usage: 4.4 MB


### 4. Generating rows with irrelevant skills

In [470]:
N = 50_000

job_ids = job_skill_match['job_id'].unique().tolist()
skill_ids = job_skill_match['skill_id'].unique().tolist()

positive_pairs = set(zip(job_skill_match['job_id'], job_skill_match['skill_id']))
negative_pairs = set()

while len(negative_pairs) < N:

    jobs_batch = random.choices(job_ids, k=1_000)
    skills_batch = random.choices(skill_ids, k=1_000)

    negative_pairs.update( (j, s) for j, s in zip(jobs_batch, skills_batch) if (j, s) not in positive_pairs)

negative_pairs = set(list(negative_pairs)[:N])

In [471]:
## add rows with irrelevant skills to train dataframe

negatives = pd.DataFrame(list(negative_pairs), columns=['job_id', 'skill_id'])
negatives = pd.merge(negatives, jobs[['job_id', 'job_name']], on='job_id', how='left')
negatives = pd.merge(negatives, skills[['skill_id', 'skill_name']], on='skill_id', how='left')
negatives['label'] = 'irrelevant'

train = pd.concat([train, negatives], ignore_index=True)

In [472]:
train['label'] = train['label'].map({ 'essential': 2, 'optional': 1, 'irrelevant': 0 })

In [473]:
train

,job_id,skill_id,label,job_name,skill_name
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,2,technical director,promote health and safety
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,2,technical director,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,2,technical director,negotiate health and safety issues with third ...
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,2,technical director,theatre techniques
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,2,technical director,coordinate technical teams in artistic product...
...,...,...,...,...,...
164694,http://data.europa.eu/esco/occupation/8099a96a...,http://data.europa.eu/esco/skill/2aeadb90-7ddc...,0,import export specialist in live animals,perform dances
164695,http://data.europa.eu/esco/occupation/85fdd1cb...,http://data.europa.eu/esco/skill/a8da58c7-42fc...,0,textile quality inspector,nanotechnology
164696,http://data.europa.eu/esco/occupation/3c1b7871...,http://data.europa.eu/esco/skill/f54cb941-6e9b...,0,musical instrument technician,manage aftersales processes to comply with bus...
164697,http://data.europa.eu/esco/occupation/76abbb82...,http://data.europa.eu/esco/skill/98d4819d-14c0...,0,commissioning engineer,define artistic performance concepts


In [474]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 164699 entries, 0 to 164698
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   job_id      164699 non-null  str  
 1   skill_id    164699 non-null  str  
 2   label       164699 non-null  int64
 3   job_name    164699 non-null  str  
 4   skill_name  164699 non-null  str  
dtypes: int64(1), str(4)
memory usage: 6.3 MB


### 5. Saving the training data

In [475]:
train[['job_name', 'skill_name', 'label']].to_csv('./data/processed/train.csv', index=False)

## Test data preparation 

### 1. Load the necessary files

In [476]:
job_skill_match_test = pd.read_csv( './data/raw/TaskB/validation/qrels.tsv', header=None, sep="\t", 
                                   names=['job_id', '', 'skill_id', 'label'] )

job_skill_match_test = job_skill_match_test[['job_id', 'skill_id', 'label']]

In [477]:
jobs_test = pd.read_csv( './data/raw/TaskB/validation/queries', sep='\t' )
jobs_test.rename(columns={'jobtitle': 'job_name', 'q_id': 'job_id'}, inplace=True)

In [478]:
skills_test = pd.read_csv( './data/raw/TaskB/validation/corpus_elements', sep='\t' )
skills_test.rename(columns={'c_id': 'skill_id'}, inplace=True)
skills_test['skill_aliases'] = skills_test['skill_aliases'].apply(ast.literal_eval)

### 2. Check the data

#### 2.1 job_skill_match_test

In [479]:
job_skill_match_test.head()

,job_id,skill_id,label
0,dev_qb_jt_1,dev_cb_sk_64,1
1,dev_qb_jt_1,dev_cb_sk_197,1
2,dev_qb_jt_1,dev_cb_sk_202,2
3,dev_qb_jt_1,dev_cb_sk_204,1
4,dev_qb_jt_1,dev_cb_sk_261,1


In [480]:
job_skill_match_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 56418 entries, 0 to 56417
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   job_id    56418 non-null  str  
 1   skill_id  56418 non-null  str  
 2   label     56418 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.3 MB


In [481]:
job_skill_match_test.duplicated().sum()

np.int64(0)

In [482]:
job_skill_match_test['job_id'].unique().size

304

In [483]:
job_skill_match_test['skill_id'].unique().size

9052

In [484]:
job_skill_match_test['label'].value_counts()

label
1    40257
2    16161
Name: count, dtype: int64

#### 2.2 jobs

In [485]:
jobs_test.head()

,job_id,job_name
0,dev_qb_jt_1,corporate governance analyst
1,dev_qb_jt_2,logistics business analyst
2,dev_qb_jt_3,operations planning analyst
3,dev_qb_jt_4,real estate data analyst
4,dev_qb_jt_5,security analyst ii


In [486]:
jobs_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   job_id    304 non-null    str  
 1   job_name  304 non-null    str  
dtypes: str(2)
memory usage: 4.9 KB


In [487]:
jobs_test['job_id'].unique().size

304

#### 2.3 skills

In [488]:
skills_test.head()

,skill_id,esco_uri,skill_aliases
0,dev_cb_sk_0,http://data.europa.eu/esco/skill/9bc3b9ae-50d8...,"[3D body scanning technologies, technologies f..."
1,dev_cb_sk_1,http://data.europa.eu/esco/skill/97965983-0da4...,"[3D modelling, 3D designing, three-dimensional..."
2,dev_cb_sk_2,http://data.europa.eu/esco/skill/2afb2b59-c9a3...,[3D printing process]
3,dev_cb_sk_3,http://data.europa.eu/esco/skill/3e7516dc-0f7c...,[3D texturing]
4,dev_cb_sk_4,http://data.europa.eu/esco/skill/a276b1ad-c0b7...,"[5S methodology, 5S workplace organization met..."


In [489]:
skills_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 9052 entries, 0 to 9051
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   skill_id       9052 non-null   str   
 1   esco_uri       9052 non-null   str   
 2   skill_aliases  9052 non-null   object
dtypes: object(1), str(2)
memory usage: 212.3+ KB


In [490]:
skills_test['skill_id'].unique().size

9052

In [491]:
skills_test['esco_uri'].unique().size

8942

### 3. Creating the testing dataset

In [492]:
skills_test['skill_name'] = skills_test['skill_aliases'].apply( lambda x: x[0] )

In [493]:
test = pd.merge( left=job_skill_match_test, right=jobs_test, how='inner', on='job_id' )
test = pd.merge( left=test, right=skills_test[['skill_id', 'skill_name']], how='inner', on='skill_id' )

In [494]:
test.head()

,job_id,skill_id,label,job_name,skill_name
0,dev_qb_jt_1,dev_cb_sk_64,1,corporate governance analyst,address identified risks
1,dev_qb_jt_1,dev_cb_sk_197,1,corporate governance analyst,advise on corporate social responsibility
2,dev_qb_jt_1,dev_cb_sk_202,2,corporate governance analyst,advise on drafting policies
3,dev_qb_jt_1,dev_cb_sk_204,1,corporate governance analyst,advise on efficiency improvements
4,dev_qb_jt_1,dev_cb_sk_261,1,corporate governance analyst,advise on risk management


In [495]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 56418 entries, 0 to 56417
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   job_id      56418 non-null  str  
 1   skill_id    56418 non-null  str  
 2   label       56418 non-null  int64
 3   job_name    56418 non-null  str  
 4   skill_name  56418 non-null  str  
dtypes: int64(1), str(4)
memory usage: 2.2 MB


### 4. Generating rows with irrelevant skills

In [496]:
N = 40_000

job_ids = job_skill_match_test['job_id'].unique().tolist()
skill_ids = job_skill_match_test['skill_id'].unique().tolist()

positive_pairs = set(zip(job_skill_match_test['job_id'], job_skill_match_test['skill_id']))
negative_pairs = set()

while len(negative_pairs) < N:

    jobs_batch = random.choices(job_ids, k=1_000)
    skills_batch = random.choices(skill_ids, k=1_000)

    negative_pairs.update( (j, s) for j, s in zip(jobs_batch, skills_batch) if (j, s) not in positive_pairs)

negative_pairs = set(list(negative_pairs)[:N])

In [497]:
negatives = pd.DataFrame(list(negative_pairs), columns=['job_id', 'skill_id'])
negatives = pd.merge(negatives, jobs_test, on='job_id', how='left')
negatives = pd.merge(negatives, skills_test[['skill_id', 'skill_name']], on='skill_id', how='left')
negatives['label'] = 0

test = pd.concat([test, negatives], ignore_index=True)

In [498]:
test

,job_id,skill_id,label,job_name,skill_name
0,dev_qb_jt_1,dev_cb_sk_64,1,corporate governance analyst,address identified risks
1,dev_qb_jt_1,dev_cb_sk_197,1,corporate governance analyst,advise on corporate social responsibility
2,dev_qb_jt_1,dev_cb_sk_202,2,corporate governance analyst,advise on drafting policies
3,dev_qb_jt_1,dev_cb_sk_204,1,corporate governance analyst,advise on efficiency improvements
4,dev_qb_jt_1,dev_cb_sk_261,1,corporate governance analyst,advise on risk management
...,...,...,...,...,...
96413,dev_qb_jt_181,dev_cb_sk_3575,0,csr coordinator,identify preventive actions
96414,dev_qb_jt_68,dev_cb_sk_4533,0,online project manager,maintain ship brightwork
96415,dev_qb_jt_92,dev_cb_sk_4736,0,2nd assistant accountant,manage digital documents
96416,dev_qb_jt_279,dev_cb_sk_3426,0,international trade assistant,haul cement


In [499]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 96418 entries, 0 to 96417
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   job_id      96418 non-null  str  
 1   skill_id    96418 non-null  str  
 2   label       96418 non-null  int64
 3   job_name    96418 non-null  str  
 4   skill_name  96418 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.7 MB


### 5. Saving the testing data

In [500]:
test[['job_name', 'skill_name', 'label']].to_csv('./data/processed/test.csv', index=False)